# 🎓 Day 2 · Session 2
# 02C · LLM-as-Judge, Hallucination Detection & A/B Testing
## Judge quality, detect unsupported claims and choose the best configuration

## 🎯 Learning Objectives

- Use LLM-as-a-Judge
- Detect hallucinations
- Compare configurations with A/B testing
- Build a production evaluation checklist

## 🔧 Setup Note

These notebooks are designed for **VS Code, Jupyter and Google Colab**.

For API-based evaluation demos, keep your `.env` file in the same folder as the notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas numpy

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
import numpy as np
from openai import OpenAI

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.2, system_prompt="You are a helpful AI evaluation assistant.", max_tokens=800):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ LLM-as-a-Judge

LLM-as-a-Judge uses another model to evaluate output quality on custom dimensions such as accuracy, completeness, pedagogy and safety.

# 2️⃣ Judge Prompt

In [ ]:
judge_prompt_template = '''
You are an expert evaluator for academic AI systems.

Rate the answer on four dimensions from 1 to 5:
1. accuracy
2. completeness
3. student_friendliness
4. safety

Return valid JSON only:
{
  "accuracy": 1,
  "completeness": 1,
  "student_friendliness": 1,
  "safety": 1,
  "verdict": "Pass or Fail",
  "reason": "one sentence"
}

Question:
{question}

Ground Truth:
{ground_truth}

Answer:
{answer}
'''

In [ ]:
def judge_answer(question, ground_truth, answer):
    prompt = judge_prompt_template.format(question=question, ground_truth=ground_truth, answer=answer)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a strict but fair evaluator. Return JSON only."},
            {"role": "user", "content": prompt}
        ]
    )
    return json.loads(response.choices[0].message.content)

judge_answer("What is the M.Tech CSE fee?", "Rs. 50,000 per semester", "The fee is Rs. 50,000 per semester.")

# 3️⃣ Hallucination Detection

In [ ]:
hallucination_prompt_template = '''
You are a hallucination detector.

Check whether every factual claim in the answer is supported by the context.

Return valid JSON only:
{
  "has_hallucination": false,
  "unsupported_claims": [],
  "confidence": 0.0,
  "explanation": "short explanation"
}

Retrieved Context:
{context}

Model Answer:
{answer}
'''

def detect_hallucination(context, answer):
    prompt = hallucination_prompt_template.format(context=context, answer=answer)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You detect unsupported claims. Return JSON only."},
            {"role": "user", "content": prompt}
        ]
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
context = "M.Tech CSE tuition fee is Rs. 50,000 per semester. GATE students get 50% scholarship."
good_answer = "The M.Tech CSE tuition fee is Rs. 50,000 per semester. GATE students get 50% scholarship."
bad_answer = "The M.Tech CSE tuition fee is Rs. 75,000 per semester. GATE students get 60% scholarship."

print("Good Answer Check:")
print(detect_hallucination(context, good_answer))

print("\nBad Answer Check:")
print(detect_hallucination(context, bad_answer))

# 4️⃣ A/B Testing

In [ ]:
ab_results = pd.DataFrame([
    {"Configuration": "A", "Prompt": "Baseline prompt", "Faithfulness": 0.78, "Context Recall": 0.61, "Precision": 0.80, "Pass Rate": 0.68},
    {"Configuration": "B", "Prompt": "Strict context-only prompt", "Faithfulness": 0.91, "Context Recall": 0.82, "Precision": 0.74, "Pass Rate": 0.87},
    {"Configuration": "C", "Prompt": "Longer chunks", "Faithfulness": 0.84, "Context Recall": 0.88, "Precision": 0.71, "Pass Rate": 0.75},
])
ab_results

In [ ]:
winner = ab_results.sort_values("Pass Rate", ascending=False).iloc[0]
print("Winner Configuration:", winner["Configuration"])
print("Reason: Highest pass rate with strong faithfulness.")

# 5️⃣ CI/CD for AI

In [ ]:
def rag_passes_eval(metrics, min_faithfulness=0.80, min_pass_rate=0.80):
    return metrics["faithfulness"] >= min_faithfulness and metrics["pass_rate"] >= min_pass_rate

candidate_metrics = {"faithfulness": 0.91, "pass_rate": 0.87}
rag_passes_eval(candidate_metrics)

# 6️⃣ Production Checklist

In [ ]:
pd.DataFrame([
    {"Area": "Golden Test Set", "Check": "20-50 representative examples"},
    {"Area": "RAGAS", "Check": "Faithfulness and recall above threshold"},
    {"Area": "Judge", "Check": "Pass rate above target"},
    {"Area": "Hallucination", "Check": "Unsupported claims flagged"},
    {"Area": "Out-of-scope", "Check": "Model refuses correctly"},
    {"Area": "Monitoring", "Check": "Track drift over time"},
])

# 🧪 Faculty Exercise

Design your own judge rubric for an AI teaching assistant.

Include at least:
1. Accuracy
2. Completeness
3. Pedagogy
4. Safety

# ✅ Summary

LLM-as-a-Judge, hallucination detection and A/B testing help convert opinions into deployment decisions.